##### Token Usage:

Total Tokens = Input Tokens + Output Tokens

In a chat, every request includes:

* System prompt
* Previous conversation (depending on what you send)
* Current user message


###### Why can't one tokenizer work for all models?

Each model is trained with its own vocabulary and tokenization algorithm.


| Model Provider                        | Tokenizer                                                                                                |
| ------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| OpenAI (GPT-4.1, GPT-4o, GPT-5, etc.) | `tiktoken`                                                                                               |
| Anthropic (Claude)                    | Anthropic's own tokenizer (API returns usage; no official standalone tokenizer equivalent to `tiktoken`) |
| Google (Gemini)                       | Google's tokenizer (via Vertex AI/Google AI APIs)                                                        |
| Meta (Llama)                          | Hugging Face `transformers` tokenizer or `sentencepiece`                                                 |
| Mistral                               | Hugging Face `AutoTokenizer`                                                                             |
| DeepSeek                              | Hugging Face `AutoTokenizer`                                                                             |
| Qwen                                  | Hugging Face `AutoTokenizer`                                                                             |
| Gemma                                 | Hugging Face `AutoTokenizer`                                                                             |


##### Cached tokens:

1) First request

            System Prompt:
            You are a Bank Support Assistant.

            Conversation:
            User: Hi
            Assistant: Hello! How can I help?

            User:
            What is a savings account?

            System Prompt          100 tokens
            Conversation            30 tokens
            User Question            8 tokens
            -----------------------------
            Input Tokens           138

            nothing is cached.

            Input Tokens: 138
            Cached Tokens: 0

2) Second request

            System Prompt:
            You are a Bank Support Assistant.

            Conversation:
            User: Hi
            Assistant: Hello! How can I help?
            User: What is a savings account?
            Assistant: ...

            User:
            How do I open one?

            System Prompt          100
            Conversation            80
            New User Message         6
            ----------------------------
            Input Tokens          186

            Cached Tokens        180
            New Tokens             6

# Prompt Caching Internals

## Overview

Prompt caching allows an LLM provider to **reuse previously computed Transformer Key-Value (KV) cache** for an identical prompt prefix, reducing latency and compute.

---

## Step 1: User sends a prompt

```text
System:
You are a helpful assistant.

User:
Explain HTTPS.
```

Pipeline:

```text
Text
 ↓
Tokenizer
 ↓
Token IDs
 ↓
Embeddings
 ↓
Transformer Layers
 ↓
Output
```

---

## Step 2: Tokens become embeddings

```text
"You" → Token ID → Embedding Vector
```

Every token is converted into a dense vector.

---

## Step 3: Every transformer layer computes Q, K, and V

```text
Embedding
   │
   ├──> Query (Q)
   ├──> Key (K)
   └──> Value (V)
```

The attention mechanism uses these vectors.

---

## Step 4: Build the KV Cache

Example:

| Token | Key | Value |
|------|------|--------|
| You | K1 | V1 |
| are | K2 | V2 |
| a | K3 | V3 |
| helpful | K4 | V4 |
| assistant | K5 | V5 |

These Key and Value tensors are stored.

```text
KV Cache

K1 V1
K2 V2
K3 V3
K4 V4
K5 V5
```

---

## Step 5: A second request arrives

```text
System:
You are a helpful assistant.

User:
How does HTTPS work?
```

The system prompt is identical.

---

## Step 6: Provider checks for a cached prefix

Conceptually:

```text
Prompt Prefix
      │
      ▼
Fingerprint / Cache Lookup
      │
      ▼
Cache Hit?
```

If the prefix matches a previously cached prompt, the stored KV cache is reused.

---

## Step 7: Compute only new tokens

Instead of recomputing:

```text
You
are
a
helpful
assistant
How
does
HTTPS
work
```

The provider reuses:

```text
You
are
a
helpful
assistant
```

and computes only:

```text
How
does
HTTPS
work
```

New KV entries:

```text
K6 V6
K7 V7
K8 V8
K9 V9
```

The final KV cache is:

```text
Old KV Cache
+
New KV Cache
```

---

## Step 8: Continue generation

The model resumes attention using the combined KV cache and generates the response.

---

# Why is it faster?

Without caching:

```text
3000 cached-prefix tokens
+
20 new tokens

= 3020 tokens processed
```

With caching:

```text
Reuse: 3000 tokens

Compute: 20 tokens
```

This dramatically reduces GPU computation and latency.

---

# Why must the prefix be identical?

Even a small change:

```text
You are a helpful assistant.
```

to

```text
You are a very helpful assistant.
```

changes tokenization and the resulting Key/Value tensors.

The previous cache is no longer valid beyond the changed point.

---

# KV Cache vs Prompt Cache

| KV Cache | Prompt Cache |
|-----------|--------------|
| Lives during inference | Lives across API requests |
| Stores Key/Value tensors | Reuses stored KV cache for matching prompt prefixes |
| Used by every Transformer | Implemented by the model provider |
| Speeds up token generation | Speeds up repeated prompts |

---

# Interview Summary

> Prompt caching works by storing the Transformer's Key-Value (KV) cache for a previously processed prompt prefix. When a future request begins with the same prefix, the provider reuses the cached KV tensors instead of recomputing them. The model only computes attention for the new tokens appended to the prompt, reducing latency, GPU compute, and often cost.


##### Semantic Cache [We should setup in our application]

* User is aksing a question 1 [How to reset the password?] and then question 2
[I forgot my password, how should i rest it?]
* If we observe the questions the words might be changing but the meaning is similar right?
* If we do cosine similarity we would get the values close to each other of both questions.
* In that case, the second question doesn't even required to give the input to LLM ot can given the catched answer.
* We can reduce the token usage.

                        User Question
                              │
                              ▼
                        Embedding Model
                              │
                              ▼
                        Query Vector
                              │
                              ▼
                        Vector Database Cache
                              │
               ┌──────────────┴──────────────┐
               │                             │
         Similar?                      Not Similar
               │                             │
               ▼                             ▼
         Cached Answer                 Call LLM
               │                             │
               └──────────────┬──────────────┘
                              ▼
                        Return Answer

### Prompt Cache vs Semantic Cache

| Prompt Cache                     | Semantic Cache                                  |
| -------------------------------- | ----------------------------------------------- |
| Compares exact prompt prefixes   | Compares meaning using embeddings               |
| Uses cached KV tensors           | Uses cached question-answer pairs               |
| Implemented by the LLM provider  | Usually implemented by the application          |
| Speeds up inference              | Can avoid inference entirely                    |
| Requires identical prompt prefix | Works with different wording but similar intent |


